# Perspective + OCR Dual-CNN Model

Two-stage pipeline:
1. **PerspectiveNet** — CNN that predicts the 3×3 perspective matrix from the warped image
2. **Spatial Transformer** — applies the predicted matrix to straighten the card (differentiable)
3. **OCR Net** — CNN that reads the 16 digits from the straightened card

**Loss** = MSE(predicted_matrix, GT_matrix) + CrossEntropy(predicted_digits, GT_digits)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import os
import json
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

## Dataset
Loads perspective images, their ground-truth 3×3 matrices, and the card-number labels.

In [ ]:
class PerspectiveOCRDataset(Dataset):
    """
    Dataset for the dual-CNN model.
    Each sample contains:
      - The warped image (perspective_dataset_with_cards/images/)
      - The 3x3 perspective matrix (perspective_dataset_with_cards/matrices/)
      - The card number label (perspective_dataset_with_cards/labels/)
    """
    def __init__(self, img_dir, matrix_dir, label_dir, img_size=(256, 256)):
        self.img_dir = img_dir
        self.matrix_dir = matrix_dir
        self.label_dir = label_dir

        self.image_files = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])

        self.chars = "0123456789"
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.idx_to_char = {i: c for i, c in enumerate(self.chars)}

        self.img_size = img_size

        self.transform = transforms.Compose([
            transforms.Resize(img_size),
            
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def encode_label(self, text):
        clean = text.replace(' ', '').strip()
        encoded = [self.char_to_idx[c] for c in clean if c in self.char_to_idx]
        return torch.tensor(encoded, dtype=torch.long)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        stem = os.path.splitext(img_name)[0]

        # Load image
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)

        # Load 3x3 perspective matrix
        mat_path = os.path.join(self.matrix_dir, f"{stem}.npy")
        matrix = np.load(mat_path).astype(np.float32)  # (3, 3)
        matrix_tensor = torch.from_numpy(matrix)

        # Load label
        lbl_path = os.path.join(self.label_dir, f"{stem}.txt")
        with open(lbl_path, 'r') as f:
            text_label = f.read().strip()
        label = self.encode_label(text_label)

        return {
            'image': image,          # (3, H, W)
            'matrix': matrix_tensor, # (3, 3)
            'label': label           # (16,) digits
        }

## Load Dataset

In [ ]:
# Set to True if running on Colab with downloaded data
USE_COLAB = False

if USE_COLAB:
    !pip install gdown
    # Download and unzip your data here
    IMG_DIR = "/content/perspective_dataset_with_cards/images"
    MAT_DIR = "/content/perspective_dataset_with_cards/matrices"
    LBL_DIR = "/content/perspective_dataset_with_cards/labels"
else:
    IMG_DIR = "perspective_dataset_with_cards/images"
    MAT_DIR = "perspective_dataset_with_cards/matrices"
    LBL_DIR = "perspective_dataset_with_cards/labels"

IMG_SIZE = (256, 256)
dataset = PerspectiveOCRDataset(IMG_DIR, MAT_DIR, LBL_DIR, img_size=IMG_SIZE)
print(f"Dataset size: {len(dataset)} samples")

sample = dataset[0]
print(f"Image shape: {sample['image'].shape}")
print(f"Matrix shape: {sample['matrix'].shape}")
print(f"Label: {sample['label']}")

# Compute per-element normalization statistics for perspective matrices
matrix_mean, matrix_std = compute_matrix_stats(MAT_DIR)
print(f"\nMatrix per-element mean:\n{matrix_mean}")
print(f"\nMatrix per-element std:\n{matrix_std}")

## Model Architecture

### PerspectiveNet
CNN that predicts the 9 values of the 3×3 perspective matrix.

### Differentiable Spatial Transformer
Applies the predicted matrix to warp/straighten the image (using `torch.nn.functional.grid_sample`).

### OCRNet  
CNN that reads 16 digits from the straightened card image (same architecture as the existing MOCNN).

In [ ]:
class PerspectiveNet(nn.Module):
    """
    CNN to predict the 3x3 perspective matrix from a warped card image.
    Output: 9 values (flattened 3x3 matrix)
    """
    def __init__(self, in_channels=3):
        super(PerspectiveNet, self).__init__()

        self.features = nn.Sequential(
            # Block 1: 256x256 -> 128x128
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 2: 128x128 -> 64x64
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3: 64x64 -> 32x32
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 4: 32x32 -> 16x16
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 5: 16x16 -> 8x8
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 6: 8x8 -> 4x4
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.regressor = nn.Sequential(
            nn.Linear(512 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 9)  # 3x3 matrix = 9 values
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.regressor(x)
        return x.view(-1, 3, 3)  # Reshape to (B, 3, 3)

In [ ]:
class DifferentiableSpatialTransformer(nn.Module):
    """
    Applies a 3x3 perspective matrix to warp an image using differentiable grid_sample.
    This makes the entire pipeline end-to-end trainable.
    """
    def __init__(self, output_size=(128, 64)):
        super(DifferentiableSpatialTransformer, self).__init__()
        self.output_h, self.output_w = output_size

    def forward(self, image, matrix):
        """
        Args:
            image: (B, C, H, W) input warped image
            matrix: (B, 3, 3) predicted perspective matrix
        Returns:
            straightened: (B, C, out_H, out_W) the straightened card image
        """
        B, C, H, W = image.shape
        device = image.device

        # Create output grid in pixel coordinates (destination space)
        # The matrix maps FROM source (warped) TO destination (straightened)
        # We need the inverse: map FROM destination TO source for grid_sample
        out_h, out_w = self.output_h, self.output_w

        # Create normalized grid for output image [-1, 1]
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(0, out_h - 1, out_h, device=device),
            torch.linspace(0, out_w - 1, out_w, device=device),
            indexing='ij'
        )

        # Stack to homogeneous coordinates (out_h*out_w, 3)
        ones = torch.ones_like(grid_x)
        grid_flat = torch.stack([grid_x.flatten(), grid_y.flatten(), ones.flatten()], dim=0)  # (3, N)

        # The GT matrix maps warped_quad -> straightened_rect
        # Inverse maps straightened_rect -> warped_quad (source coords for sampling)
        matrix_inv = torch.inverse(matrix)  # (B, 3, 3)

        # Apply inverse perspective to get source coordinates
        grid_batch = grid_flat.unsqueeze(0).expand(B, -1, -1)  # (B, 3, N)
        src_coords = torch.bmm(matrix_inv, grid_batch)  # (B, 3, N)

        # Dehomogenize
        src_x = src_coords[:, 0, :] / (src_coords[:, 2, :] + 1e-8)
        src_y = src_coords[:, 1, :] / (src_coords[:, 2, :] + 1e-8)

        # Normalize to [-1, 1] for grid_sample (based on input image size)
        src_x_norm = 2.0 * src_x / (W - 1) - 1.0
        src_y_norm = 2.0 * src_y / (H - 1) - 1.0

        # Reshape to grid format (B, out_h, out_w, 2)
        grid = torch.stack([src_x_norm, src_y_norm], dim=-1)  # (B, N, 2)
        grid = grid.view(B, out_h, out_w, 2)

        # Sample from input image
        straightened = F.grid_sample(image, grid, mode='bilinear', padding_mode='zeros', align_corners=True)

        return straightened

In [ ]:
class OCRNet(nn.Module):
    """
    CNN for reading 16 digits from the straightened card.
    Input: (B, 3, 64, 128) straightened card image
    Output: (B, 16, 10) logits for each digit position
    """
    def __init__(self, in_channels=3, num_classes=10):
        super(OCRNet, self).__init__()

        self.features = nn.Sequential(
            # Block 1: 64x128 -> 32x64
            nn.Conv2d(in_channels, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 2: 32x64 -> 16x32
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3: 16x32 -> 8x16
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 4: 8x16 -> 4x8
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 5: 4x8 -> 2x4
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 6: 2x4 -> 1x16 (via adaptive pool)
            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 16))  # Collapse height, keep 16 positions
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)       # (B, 512, 1, 16)
        x = x.squeeze(2)            # (B, 512, 16)
        x = x.permute(0, 2, 1)      # (B, 16, 512)
        x = self.classifier(x)      # (B, 16, 10)
        return x

In [ ]:
class PerspectiveOCRModel(nn.Module):
    """
    Combined end-to-end model:
    1. PerspectiveNet predicts 3x3 matrix
    2. SpatialTransformer applies it to straighten the card
    3. OCRNet reads the digits from the straightened image
    """
    def __init__(self, card_output_size=(64, 128)):
        super(PerspectiveOCRModel, self).__init__()

        self.perspective_net = PerspectiveNet(in_channels=3)
        self.spatial_transformer = DifferentiableSpatialTransformer(output_size=card_output_size)
        self.ocr_net = OCRNet(in_channels=3, num_classes=10)

    def forward(self, x):
        # Step 1: Predict perspective matrix
        pred_matrix = self.perspective_net(x)  # (B, 3, 3)

        # Step 2: Apply matrix to straighten the card
        straightened = self.spatial_transformer(x, pred_matrix)  # (B, 3, 64, 128)

        # Step 3: OCR on the straightened card
        digit_logits = self.ocr_net(straightened)  # (B, 16, 10)

        return pred_matrix, straightened, digit_logits

## Collate Function & Helpers

In [ ]:
def perspective_ocr_collate_fn(batch):
    """Custom collate: stack images & matrices, pad labels."""
    batch = [item for item in batch if item is not None]

    images = torch.stack([item['image'] for item in batch], 0)
    matrices = torch.stack([item['matrix'] for item in batch], 0)
    labels = pad_sequence([item['label'] for item in batch], batch_first=True, padding_value=0)

    return images, matrices, labels


def compute_matrix_stats(matrix_dir):
    """
    Compute per-element mean and std across all 3x3 perspective matrices.
    First normalizes each matrix by M[2,2], then computes statistics.
    Returns tensors of shape (3, 3) for mean and std.
    """
    files = sorted([f for f in os.listdir(matrix_dir) if f.endswith('.npy')])
    all_matrices = []
    for f in files:
        m = np.load(os.path.join(matrix_dir, f)).astype(np.float32)
        # Normalize by M[2,2] so last element is 1
        m = m / (m[2, 2] + 1e-8)
        all_matrices.append(m)

    all_matrices = np.stack(all_matrices)  # (N, 3, 3)
    mean = all_matrices.mean(axis=0)       # (3, 3)
    std = all_matrices.std(axis=0)         # (3, 3)
    std = np.maximum(std, 1e-6)            # avoid division by zero

    return torch.from_numpy(mean), torch.from_numpy(std)


def normalize_matrix(matrix, mean, std):
    """
    Per-element normalization of 3x3 perspective matrix.
    1. Divide by M[2,2] to make it scale-invariant
    2. Subtract per-element mean and divide by per-element std
    This brings all 9 elements to roughly N(0,1) range,
    so MSE loss is balanced across translation, rotation, and projective terms.
    """
    # First normalize by M[2,2]
    scale = matrix[:, 2, 2].unsqueeze(-1).unsqueeze(-1) + 1e-8
    normed = matrix / scale
    # Per-element normalization
    return (normed - mean.unsqueeze(0).to(matrix.device)) / std.unsqueeze(0).to(matrix.device)


def denormalize_matrix(normed, mean, std):
    """Reverse per-element normalization to get raw perspective matrix."""
    return normed * std.unsqueeze(0).to(normed.device) + mean.unsqueeze(0).to(normed.device)

## Training Loop

Combined loss = `alpha * MSE(pred_matrix, gt_matrix) + beta * CrossEntropy(pred_digits, gt_digits)`

In [ ]:
def train_model(model, train_loader, val_loader, matrix_mean, matrix_std,
                num_epochs=50, lr=1e-4, device='cuda', alpha=1.0, beta=1.0):
    """
    Train the dual-CNN model.
    alpha: weight for matrix MSE loss (computed in per-element-normalized space)
    beta: weight for OCR cross-entropy loss
    matrix_mean, matrix_std: (3,3) tensors for per-element normalization
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    mse_criterion = nn.MSELoss()
    ce_criterion = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_loss': [],
               'train_mse': [], 'val_mse': [],
               'train_ce': [], 'val_ce': []}

    print(f"Training on {device} | alpha={alpha}, beta={beta}")
    print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'MSE':>10} {'CE':>10}")
    print("-" * 55)

    for epoch in range(num_epochs):
        # --- Training ---
        model.train()
        train_loss, train_mse, train_ce = 0.0, 0.0, 0.0

        for images, gt_matrices, gt_labels in train_loader:
            images = images.to(device)
            gt_matrices = gt_matrices.to(device)
            gt_labels = gt_labels.to(device)

            # Forward
            pred_matrix, straightened, digit_logits = model(images)

            # Normalize matrices with per-element stats for balanced MSE
            pred_norm = normalize_matrix(pred_matrix, matrix_mean, matrix_std)
            gt_norm = normalize_matrix(gt_matrices, matrix_mean, matrix_std)

            # MSE loss on normalized perspective matrix
            loss_mse = mse_criterion(pred_norm, gt_norm)

            # Cross-entropy loss on digit predictions
            num_digits = min(digit_logits.size(1), gt_labels.size(1))
            logits_for_ce = digit_logits[:, :num_digits, :]
            labels_for_ce = gt_labels[:, :num_digits]
            loss_ce = ce_criterion(logits_for_ce.reshape(-1, 10), labels_for_ce.reshape(-1))

            # Combined loss
            loss = alpha * loss_mse + beta * loss_ce

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            train_loss += loss.item()
            train_mse += loss_mse.item()
            train_ce += loss_ce.item()

        avg_train = train_loss / len(train_loader)
        avg_train_mse = train_mse / len(train_loader)
        avg_train_ce = train_ce / len(train_loader)

        # --- Validation ---
        model.eval()
        val_loss, val_mse, val_ce = 0.0, 0.0, 0.0

        with torch.no_grad():
            for images, gt_matrices, gt_labels in val_loader:
                images = images.to(device)
                gt_matrices = gt_matrices.to(device)
                gt_labels = gt_labels.to(device)

                pred_matrix, straightened, digit_logits = model(images)

                pred_norm = normalize_matrix(pred_matrix, matrix_mean, matrix_std)
                gt_norm = normalize_matrix(gt_matrices, matrix_mean, matrix_std)

                loss_mse = mse_criterion(pred_norm, gt_norm)

                num_digits = min(digit_logits.size(1), gt_labels.size(1))
                logits_for_ce = digit_logits[:, :num_digits, :]
                labels_for_ce = gt_labels[:, :num_digits]
                loss_ce = ce_criterion(logits_for_ce.reshape(-1, 10), labels_for_ce.reshape(-1))

                loss = alpha * loss_mse + beta * loss_ce

                val_loss += loss.item()
                val_mse += loss_mse.item()
                val_ce += loss_ce.item()

        avg_val = val_loss / len(val_loader)
        avg_val_mse = val_mse / len(val_loader)
        avg_val_ce = val_ce / len(val_loader)

        scheduler.step(avg_val)

        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['train_mse'].append(avg_train_mse)
        history['val_mse'].append(avg_val_mse)
        history['train_ce'].append(avg_train_ce)
        history['val_ce'].append(avg_val_ce)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"{epoch+1:>6} {avg_train:>12.4f} {avg_val:>12.4f} {avg_val_mse:>10.4f} {avg_val_ce:>10.4f}")

    return model, history

## Plot Training History

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Total loss
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('Total Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True)

    # MSE loss (matrix)
    axes[1].plot(history['train_mse'], label='Train MSE')
    axes[1].plot(history['val_mse'], label='Val MSE')
    axes[1].set_title('Matrix MSE Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True)

    # CE loss (OCR)
    axes[2].plot(history['train_ce'], label='Train CE')
    axes[2].plot(history['val_ce'], label='Val CE')
    axes[2].set_title('OCR CrossEntropy Loss')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()

## Train

In [ ]:
# --- Configuration ---
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
EPOCHS = 50
ALPHA = 1.0   # Weight for MSE (matrix) loss
BETA = 1.0    # Weight for CrossEntropy (OCR) loss
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Split dataset ---
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=perspective_ocr_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=perspective_ocr_collate_fn)

print(f"Train: {train_size}, Val: {val_size}")
print(f"Device: {DEVICE}")

# --- Create model ---
model = PerspectiveOCRModel(card_output_size=(64, 128))
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# --- Train ---
model, history = train_model(
    model, train_loader, val_loader,
    matrix_mean=matrix_mean, matrix_std=matrix_std,
    num_epochs=EPOCHS, lr=LEARNING_RATE,
    device=DEVICE, alpha=ALPHA, beta=BETA
)

In [ ]:
plot_history(history)

## Evaluate Accuracy

In [ ]:
def evaluate_accuracy(model, val_loader, device, matrix_mean, matrix_std, num_digits=16):
    """
    Evaluate strict sequence accuracy (all 16 digits must be correct).
    Also reports per-digit accuracy and matrix MSE (in normalized space).
    """
    model.eval()
    model = model.to(device)

    correct_sequences = 0
    correct_digits = 0
    total_digits = 0
    total_samples = 0
    total_mse = 0.0

    mse_criterion = nn.MSELoss()

    with torch.no_grad():
        for images, gt_matrices, gt_labels in val_loader:
            images = images.to(device)
            gt_matrices = gt_matrices.to(device)
            gt_labels = gt_labels.to(device)

            pred_matrix, straightened, digit_logits = model(images)

            # Matrix MSE in normalized space
            pred_norm = normalize_matrix(pred_matrix, matrix_mean, matrix_std)
            gt_norm = normalize_matrix(gt_matrices, matrix_mean, matrix_std)
            total_mse += mse_criterion(pred_norm, gt_norm).item()

            # Digit accuracy
            nd = min(digit_logits.size(1), gt_labels.size(1), num_digits)
            preds = digit_logits[:, :nd, :].argmax(dim=2)  # (B, nd)
            targets = gt_labels[:, :nd]

            matches = preds.eq(targets)
            correct_sequences += matches.all(dim=1).sum().item()
            correct_digits += matches.sum().item()
            total_digits += matches.numel()
            total_samples += images.size(0)

    seq_acc = 100.0 * correct_sequences / total_samples
    digit_acc = 100.0 * correct_digits / total_digits
    avg_mse = total_mse / len(val_loader)

    print(f"{'='*40}")
    print(f"Total Samples: {total_samples}")
    print(f"Sequence Accuracy: {seq_acc:.2f}% ({correct_sequences}/{total_samples})")
    print(f"Per-Digit Accuracy: {digit_acc:.2f}%")
    print(f"Avg Matrix MSE (normalized): {avg_mse:.6f}")
    print(f"{'='*40}")

    return seq_acc, digit_acc, avg_mse


evaluate_accuracy(model, val_loader, DEVICE, matrix_mean, matrix_std)

## Visualize Predictions
Show the warped input, the predicted straightened card, and the GT straightened card side by side.

In [ ]:
def visualize_predictions(model, dataset, device, num_samples=6):
    model.eval()
    model = model.to(device)

    indices = np.random.choice(len(dataset), num_samples, replace=False)
    fig, axes = plt.subplots(num_samples, 3, figsize=(16, 5 * num_samples))

    denorm = lambda x: (x * 0.5 + 0.5).clamp(0, 1)

    for row, idx in enumerate(indices):
        sample = dataset[idx]
        image = sample['image'].unsqueeze(0).to(device)
        gt_matrix = sample['matrix'].unsqueeze(0).to(device)
        gt_label = sample['label']

        with torch.no_grad():
            pred_matrix, pred_straightened, digit_logits = model(image)

        # Decode predicted digits
        pred_digits = digit_logits.argmax(dim=2)[0].cpu().numpy()
        pred_text = ''.join([str(d) for d in pred_digits])
        pred_text_formatted = ' '.join([pred_text[i:i+4] for i in range(0, 16, 4)])

        # Decode GT digits
        gt_digits = gt_label.numpy()
        gt_text = ''.join([str(d) for d in gt_digits])
        gt_text_formatted = ' '.join([gt_text[i:i+4] for i in range(0, min(16, len(gt_text)), 4)])

        # GT straightening using actual matrix
        st = DifferentiableSpatialTransformer(output_size=(64, 128)).to(device)
        gt_straightened = st(image, gt_matrix)

        # Convert to displayable
        img_np = denorm(image[0]).cpu().permute(1, 2, 0).numpy()
        pred_st_np = denorm(pred_straightened[0]).cpu().permute(1, 2, 0).numpy()
        gt_st_np = denorm(gt_straightened[0]).cpu().permute(1, 2, 0).numpy()

        # Plot
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title('Input (warped)')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(gt_st_np)
        axes[row, 1].set_title(f'GT Straightened\n{gt_text_formatted}')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(pred_st_np)
        axes[row, 2].set_title(f'Predicted Straightened\n{pred_text_formatted}')
        axes[row, 2].axis('off')

    plt.tight_layout()
    plt.show()


visualize_predictions(model, dataset, DEVICE, num_samples=6)

## Save Model

In [ ]:
# Save the trained model
torch.save(model.state_dict(), 'perspective_ocr_model.pth')
print('Model saved to perspective_ocr_model.pth')

# Model summary
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Parameters: {total:,}')
print(f'Trainable Parameters: {trainable:,}')